# Day 3: Unsupervised Learning, Topic Models, and Scaling

Today is about discovering structure without document labels. The main classroom question is not just whether a model runs, but whether the output can be interpreted and validated.

By the end you should be able to:

1. Explain k-means and hierarchical clustering as unsupervised measurement tools.
2. Simulate text-like clusters from word-generating processes.
3. Show why an algorithmic optimum can disagree with the data-generating structure.
4. Fit and inspect a topic model.
5. Compare topic counts and topic stability.
6. Relate topic output to metadata with STM-like prevalence regressions.
7. Use SVD as a lightweight illustration of latent document scaling.
8. Write a short validation memo.


In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

os.environ.setdefault('LOKY_MAX_CPU_COUNT', '4')

from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import LatentDirichletAllocation, TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from scipy.cluster.hierarchy import dendrogram, linkage

pd.set_option('display.max_colwidth', 140)

In [ ]:
from pathlib import Path


def find_data_dir():
    candidates = [Path('../data'), Path('materials/data'), Path('data')]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not find the course data directory.')


DATA_DIR = find_data_dir()
DATA_DIR

## spaCy setup

The notebooks use spaCy for tokenization and preprocessing. If `en_core_web_sm` is installed, the same setup also shows POS tags, lemmas, and dependency parses. If not, the notebooks still run with a blank English tokenizer.

In [ ]:
import spacy

try:
    nlp = spacy.load('en_core_web_sm')
    print('Loaded spaCy model: en_core_web_sm')
except OSError:
    nlp = spacy.blank('en')
    nlp.add_pipe('sentencizer')
    print('Using spacy.blank("en"). Install en_core_web_sm for POS tags, lemmas, and dependency parses.')


def spacy_preprocess(text, remove_stop=True, keep_alpha=True, min_len=2):
    """Tokenize and lightly preprocess text with spaCy."""
    doc = nlp.make_doc(str(text))
    tokens = []
    for token in doc:
        if keep_alpha and not token.is_alpha:
            continue
        if remove_stop and token.is_stop:
            continue
        term = token.text.lower()
        if len(term) >= min_len:
            tokens.append(term)
    return tokens


def spacy_analyzer(text):
    return spacy_preprocess(text)


def spacy_analyzer_with_bigrams(text):
    tokens = spacy_preprocess(text)
    bigrams = [tokens[i] + '_' + tokens[i + 1] for i in range(len(tokens) - 1)]
    return tokens + bigrams


def spacy_token_table(text):
    """Show tokenization, preprocessing flags, and parses when a full model is available."""
    doc = nlp(str(text))
    rows = []
    for token in doc:
        rows.append({
            'text': token.text,
            'lemma': token.lemma_ or '(model needed)',
            'pos': token.pos_ or '(model needed)',
            'dep': token.dep_ or '(model needed)',
            'is_alpha': token.is_alpha,
            'is_stop': token.is_stop,
            'kept_by_preprocess': token.text.lower() in spacy_preprocess(token.text, remove_stop=True)
        })
    return pd.DataFrame(rows)

## 1. Clustering intuition with toy text

K-means partitions observations into clusters by alternating between assignment and centroid updates. We start with short simulated documents made from the words `banana`, `chocolate`, and `vanilla`, then convert those texts into counts. The plot is simple because we only visualize the banana and chocolate counts.


In [ ]:
toy_rng = np.random.default_rng(2026)
toy_vocab = np.array(['banana', 'chocolate', 'vanilla'])
toy_profiles = {
    'banana-heavy': np.array([0.72, 0.12, 0.16]),
    'chocolate-heavy': np.array([0.12, 0.72, 0.16]),
}

toy_rows = []
for style, probs in toy_profiles.items():
    for doc_number in range(4):
        words = toy_rng.choice(toy_vocab, size=12, p=probs)
        toy_rows.append({
            'doc_id': f'{style}_{doc_number}',
            'true_style': style,
            'text': ' '.join(words),
        })

toy_docs = pd.DataFrame(toy_rows)
toy_vectorizer = CountVectorizer(vocabulary=toy_vocab)
toy_counts = pd.DataFrame(
    toy_vectorizer.fit_transform(toy_docs['text']).toarray(),
    columns=toy_vectorizer.get_feature_names_out(),
)

toy = pd.concat([toy_docs, toy_counts], axis=1)

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
toy['cluster'] = kmeans.fit_predict(toy[['banana', 'chocolate']])

toy[['doc_id', 'true_style', 'text', 'banana', 'chocolate', 'vanilla', 'cluster']]


In [ ]:
plt.figure(figsize=(5.5, 4.2))
sns.scatterplot(
    data=toy,
    x='banana',
    y='chocolate',
    hue='cluster',
    style='true_style',
    s=90,
    palette='tab10',
)
plt.scatter(
    kmeans.cluster_centers_[:, 0],
    kmeans.cluster_centers_[:, 1],
    marker='X',
    s=180,
    color='black',
    label='centers',
)
plt.xlabel('banana count')
plt.ylabel('chocolate count')
plt.title('K-means on counts derived from toy texts')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()


## 1a. Visualizing k-means iterations

K-means is easier to understand as an iterative algorithm. Each round assigns every point to the nearest centroid, then recomputes each centroid as the mean of its assigned points:

$$
c_i = \arg\min_k \lVert x_i - \mu_k \rVert^2, \qquad \mu_k = \frac{1}{|C_k|}\sum_{i\in C_k}x_i.
$$

The labels themselves are arbitrary; what matters is the partition of points.


In [ ]:
toy_points = toy[['banana', 'chocolate']].to_numpy(dtype=float)
initial_centers = np.array([[8.0, 1.0], [0.0, 7.0]])


def assign_to_centers(points, centers):
    distances = np.linalg.norm(points[:, None, :] - centers[None, :, :], axis=2)
    return distances.argmin(axis=1)


def update_centers(points, labels, n_clusters):
    centers = []
    for cluster_id in range(n_clusters):
        members = points[labels == cluster_id]
        centers.append(members.mean(axis=0))
    return np.vstack(centers)


def kmeans_trace(points, centers, n_steps=4):
    states = []
    current_centers = centers.copy()
    for step in range(n_steps):
        labels = assign_to_centers(points, current_centers)
        states.append({
            'step': step,
            'centers': current_centers.copy(),
            'labels': labels.copy(),
        })
        current_centers = update_centers(points, labels, len(current_centers))
    return states


trace = kmeans_trace(toy_points, initial_centers, n_steps=4)

fig, axes = plt.subplots(1, len(trace), figsize=(14, 3.5), sharex=True, sharey=True)
colors = np.array(['#4C72B0', '#DD8452'])

for ax, state in zip(axes, trace):
    labels = state['labels']
    centers = state['centers']
    ax.scatter(toy_points[:, 0], toy_points[:, 1], c=colors[labels], s=70, edgecolor='white')
    ax.scatter(centers[:, 0], centers[:, 1], marker='X', s=160, c='black')
    for idx, (x, y) in enumerate(centers):
        ax.text(x + 0.15, y + 0.15, f'center {idx}', fontsize=8)
    ax.set_title(f'Iteration {state["step"]}')
    ax.set_xlabel('banana count')
    ax.grid(alpha=0.2)

axes[0].set_ylabel('chocolate count')
plt.suptitle('K-means alternates between assignment and centroid updates', y=1.05)
plt.tight_layout()


## 1b. Hierarchical clustering and dendrograms

Hierarchical clustering builds a nested tree of similarities. With agglomerative clustering, each document starts as its own cluster and the algorithm repeatedly merges the closest clusters. A dendrogram shows the order and distance of those merges using the same banana/chocolate count representation.


In [ ]:
toy_linkage = linkage(toy[['banana', 'chocolate']], method='ward')
toy_hierarchical = AgglomerativeClustering(n_clusters=2, linkage='ward')
toy['hierarchical_cluster'] = toy_hierarchical.fit_predict(toy[['banana', 'chocolate']])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for cluster, group in toy.groupby('hierarchical_cluster'):
    axes[0].scatter(group['banana'], group['chocolate'], label=f'cluster {cluster}', s=70)
axes[0].set_title('Agglomerative clustering result')
axes[0].set_xlabel('banana count')
axes[0].set_ylabel('chocolate count')
axes[0].legend()

dendrogram(
    toy_linkage,
    labels=[f'doc {i}' for i in toy.index],
    ax=axes[1],
    color_threshold=0,
)
axes[1].set_title('Ward-linkage dendrogram')
axes[1].set_xlabel('Toy document')
axes[1].set_ylabel('Merge distance')
plt.tight_layout()

toy


## 1c. Simulated text clusters from a Dirichlet-multinomial model

Now simulate text-like data where we know the data-generating structure. Each latent cluster has a word distribution drawn from a Dirichlet prior, and each document is a multinomial draw from one cluster's distribution. This gives us a controlled setting for asking whether clustering recovers known structure.


In [ ]:
rng = np.random.default_rng(42)

cluster_names = ['sports', 'science', 'politics']
vocabulary = np.array([
    'team', 'game', 'season', 'score', 'player',
    'space', 'orbit', 'data', 'research', 'experiment',
    'vote', 'policy', 'party', 'government', 'election',
    'today', 'people', 'issue', 'debate', 'report'
])
anchor_words = {
    'sports': ['team', 'game', 'season', 'score', 'player'],
    'science': ['space', 'orbit', 'data', 'research', 'experiment'],
    'politics': ['vote', 'policy', 'party', 'government', 'election'],
}

cluster_word_probs = {}
for cluster in cluster_names:
    alpha = np.full(len(vocabulary), 0.25)
    for word in anchor_words[cluster]:
        alpha[np.where(vocabulary == word)[0][0]] = 5.0
    cluster_word_probs[cluster] = rng.dirichlet(alpha)

simulated_rows = []
simulated_counts = []
for cluster in cluster_names:
    for doc_number in range(35):
        length = int(rng.integers(70, 150))
        counts = rng.multinomial(length, cluster_word_probs[cluster])
        simulated_rows.append({
            'doc_id': f'{cluster}_{doc_number:02d}',
            'true_cluster': cluster,
            'length': length,
        })
        simulated_counts.append(counts)

simulated_docs = pd.DataFrame(simulated_rows)
simulated_count_df = pd.DataFrame(simulated_counts, columns=vocabulary)
simulated_rates = simulated_count_df.div(simulated_count_df.sum(axis=1), axis=0)

simulated_kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
simulated_docs['kmeans_cluster'] = simulated_kmeans.fit_predict(simulated_rates)

sim_svd = TruncatedSVD(n_components=2, random_state=42)
sim_coords = sim_svd.fit_transform(simulated_rates)
simulated_docs['component_1'] = sim_coords[:, 0]
simulated_docs['component_2'] = sim_coords[:, 1]

pd.crosstab(simulated_docs['true_cluster'], simulated_docs['kmeans_cluster'])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True, sharey=True)

sns.scatterplot(
    data=simulated_docs,
    x='component_1',
    y='component_2',
    hue='true_cluster',
    s=55,
    alpha=0.85,
    ax=axes[0],
)
axes[0].set_title('Known simulated clusters')
axes[0].set_xlabel('SVD component 1')
axes[0].set_ylabel('SVD component 2')

sns.scatterplot(
    data=simulated_docs,
    x='component_1',
    y='component_2',
    hue='kmeans_cluster',
    palette='tab10',
    s=55,
    alpha=0.85,
    ax=axes[1],
)
axes[1].set_title('K-means clusters recovered from word rates')
axes[1].set_xlabel('SVD component 1')
axes[1].set_ylabel('')

plt.suptitle('Clustering text-like documents generated from multinomial word counts', y=1.04)
plt.tight_layout()


The simulated example is deliberately easier than real text: the clusters have clean anchor vocabularies and documents are generated from one cluster at a time. Real corpora are messier because documents mix themes, word meanings shift, and metadata can create structure that is not the concept we care about.


## 1d. When the k-means optimum disagrees with the DGP

In simulations we know the data-generating process. That lets us show an important warning: an internal clustering diagnostic can prefer a different number of clusters than the true number of latent groups.

Below the DGP has three groups: banana-heavy documents, chocolate-heavy documents, and mixed banana-chocolate documents. The mixed group lies between the other two. Because k-means and silhouette prefer compact, well-separated clusters, the diagnostic can choose `K=2` even though the DGP has `K=3`.

The silhouette score for document $i$ is

$$
s_i = \frac{b_i-a_i}{\max(a_i,b_i)},
$$

where $a_i$ is average distance to points in the same cluster and $b_i$ is average distance to the nearest other cluster. High silhouette means the fitted clusters are compact and separated; it does not mean the fitted `K` is the true DGP.


In [ ]:
ambiguous_rng = np.random.default_rng(1)
ambiguous_vocab = np.array([
    'banana', 'chocolate', 'vanilla', 'caramel',
    'fruit', 'dessert', 'snack', 'recipe'
])
ambiguous_profiles = {
    'banana': np.array([0.62, 0.04, 0.09, 0.03, 0.09, 0.05, 0.04, 0.04]),
    'chocolate': np.array([0.04, 0.62, 0.09, 0.08, 0.03, 0.06, 0.04, 0.04]),
    'banana_chocolate': np.array([0.31, 0.31, 0.12, 0.08, 0.04, 0.06, 0.04, 0.04]),
}

ambiguous_rows = []
for true_cluster, profile in ambiguous_profiles.items():
    for doc_number in range(45):
        doc_probs = ambiguous_rng.dirichlet(profile * 30)
        length = int(ambiguous_rng.integers(80, 150))
        counts = ambiguous_rng.multinomial(length, doc_probs)
        words = np.repeat(ambiguous_vocab, counts)
        ambiguous_rng.shuffle(words)
        ambiguous_rows.append({
            'doc_id': f'{true_cluster}_{doc_number:02d}',
            'true_cluster': true_cluster,
            'text': ' '.join(words),
        })

ambiguous_docs = pd.DataFrame(ambiguous_rows)
ambiguous_vectorizer = CountVectorizer(vocabulary=ambiguous_vocab)
ambiguous_counts = pd.DataFrame(
    ambiguous_vectorizer.fit_transform(ambiguous_docs['text']).toarray(),
    columns=ambiguous_vectorizer.get_feature_names_out(),
)
ambiguous_rates = ambiguous_counts.div(ambiguous_counts.sum(axis=1), axis=0)

ambiguous_svd = TruncatedSVD(n_components=2, random_state=42)
ambiguous_coords = ambiguous_svd.fit_transform(ambiguous_rates)
ambiguous_docs['component_1'] = ambiguous_coords[:, 0]
ambiguous_docs['component_2'] = ambiguous_coords[:, 1]

k_diagnostics = []
cluster_solutions = {}
for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = model.fit_predict(ambiguous_rates)
    cluster_solutions[k] = labels
    k_diagnostics.append({
        'k': k,
        'silhouette_higher_is_better': silhouette_score(ambiguous_rates, labels),
        'inertia_lower_is_better': model.inertia_,
        'agreement_with_true_dgp_ari': adjusted_rand_score(ambiguous_docs['true_cluster'], labels),
    })

k_diagnostics = pd.DataFrame(k_diagnostics)
best_silhouette_k = int(k_diagnostics.loc[k_diagnostics['silhouette_higher_is_better'].idxmax(), 'k'])

ambiguous_docs['kmeans_k2'] = cluster_solutions[2].astype(str)
ambiguous_docs['kmeans_k3'] = cluster_solutions[3].astype(str)

display(ambiguous_docs[['doc_id', 'true_cluster', 'text']].head(6))
display(k_diagnostics.round(3))
print(f'True DGP K = 3; best silhouette K = {best_silhouette_k}')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.lineplot(
    data=k_diagnostics,
    x='k',
    y='silhouette_higher_is_better',
    marker='o',
    ax=axes[0, 0],
)
axes[0, 0].axvline(3, color='gray', linestyle='--', label='true DGP K=3')
axes[0, 0].axvline(best_silhouette_k, color='black', linestyle=':', label=f'best silhouette K={best_silhouette_k}')
axes[0, 0].set_title('Silhouette can prefer a coarser solution')
axes[0, 0].set_xlabel('Number of fitted clusters')
axes[0, 0].set_ylabel('Silhouette score')
axes[0, 0].legend()

sns.lineplot(
    data=k_diagnostics,
    x='k',
    y='inertia_lower_is_better',
    marker='o',
    color='#DD8452',
    ax=axes[0, 1],
)
axes[0, 1].axvline(3, color='gray', linestyle='--')
axes[0, 1].set_title('Inertia always improves as K increases')
axes[0, 1].set_xlabel('Number of fitted clusters')
axes[0, 1].set_ylabel('Within-cluster sum of squares')

sns.scatterplot(
    data=ambiguous_docs,
    x='component_1',
    y='component_2',
    hue='true_cluster',
    s=45,
    alpha=0.8,
    ax=axes[1, 0],
)
axes[1, 0].set_title('True DGP: three word-generating groups')
axes[1, 0].set_xlabel('SVD component 1')
axes[1, 0].set_ylabel('SVD component 2')

sns.scatterplot(
    data=ambiguous_docs,
    x='component_1',
    y='component_2',
    hue='kmeans_k2',
    palette='tab10',
    s=45,
    alpha=0.8,
    ax=axes[1, 1],
)
axes[1, 1].set_title('K-means with the silhouette-selected K=2')
axes[1, 1].set_xlabel('SVD component 1')
axes[1, 1].set_ylabel('SVD component 2')

plt.tight_layout()


In [ ]:
print('K=2 fitted clusters versus true DGP:')
display(pd.crosstab(ambiguous_docs['true_cluster'], ambiguous_docs['kmeans_k2']))

print('K=3 fitted clusters versus true DGP:')
display(pd.crosstab(ambiguous_docs['true_cluster'], ambiguous_docs['kmeans_k3']))


The key lesson is not that silhouette is bad. It answers a specific question: which fitted partition is most compact and separated according to the chosen representation and distance. If the research question is about the true generative groups, a lower-silhouette `K=3` solution can still be substantively better than the algorithmically preferred `K=2` solution.


## 2. Topic modeling on 20 Newsgroups

We use a local classroom-sized sample from 20 Newsgroups and fit LDA with scikit-learn. The known source categories are not used for training, but they give us useful validation metadata after the unsupervised model is fit.

### Methodology formulas: clustering, topics, and scaling

K-means clustering partitions documents into $K$ groups by minimizing within-cluster squared distance:

$$\min_{C_1,\ldots,C_K}\sum_{k=1}^K\sum_{i \in C_k}\lVert x_i - \mu_k\rVert_2^2.$$

A Dirichlet-multinomial simulation first draws cluster-specific word probabilities and then draws document word counts:

$$\phi_k \sim \mathrm{Dirichlet}(\eta_k), \qquad y_i \mid z_i=k \sim \mathrm{Multinomial}(N_i, \phi_k).$$

Latent Dirichlet Allocation treats each document as a mixture of topics. A compact generative description is

$$\theta_d \sim \mathrm{Dirichlet}(\alpha), \qquad z_{dn} \sim \mathrm{Categorical}(\theta_d), \qquad w_{dn} \sim \mathrm{Categorical}(\phi_{z_{dn}}).$$

Here $\theta_{dk}$ is the share of topic $k$ in document $d$, and $\phi_{kv}$ is the probability of word $v$ under topic $k$:

$$P(w=v \mid d)=\sum_{k=1}^K \theta_{dk}\phi_{kv}.$$

An STM-like prevalence regression treats estimated topic proportions as outcomes and document metadata as covariates:

$$\hat\theta_{dk} = \alpha_k + X_d\gamma_k + \epsilon_{dk}.$$

This is not a full Structural Topic Model because the topics are estimated before the regression, but it makes the covariate logic visible.

The document scaling exercise uses a lower-dimensional representation of the document-term matrix:

$$X \approx U_k \Sigma_k V_k^\top, \qquad z_d = (U_k\Sigma_k)_{d\cdot}.$$

Interpretation should combine statistical fit with substantive validation: a topic is a useful measurement dimension only when its high-probability words, representative documents, and covariate patterns tell the same story.


In [ ]:
news = pd.read_csv(DATA_DIR / 'twenty_newsgroups_sample.csv')
news = news.dropna(subset=['text', 'category']).copy()
news['broad_area'] = np.select(
    [
        news['category'].str.startswith('talk.politics'),
        news['category'].str.startswith('sci.'),
        news['category'].str.startswith('rec.'),
        news['category'].str.startswith('comp.')
    ],
    ['politics', 'science', 'recreation', 'computing'],
    default='other'
)
news['doc_id'] = np.arange(len(news))

docs = news.reset_index(drop=True)

display(docs[['doc_id', 'category', 'broad_area', 'text']].head())
docs['category'].value_counts().sort_index()

## 2a. spaCy preprocessing for topic models

Before fitting a topic model, inspect how spaCy tokenizes a newsgroup excerpt. The topic model below uses these spaCy-preprocessed tokens.

In [ ]:
spacy_token_table(docs.loc[0, 'text'][:500]).head(25)

In [ ]:
docs['tokens'] = docs['text'].apply(spacy_analyzer)
docs[['category', 'broad_area', 'tokens']].head()

vectorizer = CountVectorizer(
    analyzer=lambda tokens: tokens,
    min_df=4,
    max_df=0.70,
    max_features=3500
)

X = vectorizer.fit_transform(docs['tokens'])
terms = vectorizer.get_feature_names_out()
X.shape

In [ ]:
n_topics = 8
lda = LatentDirichletAllocation(
    n_components=n_topics,
    learning_method='batch',
    max_iter=20,
    random_state=42
)

doc_topic = lda.fit_transform(X)
topic_word = lda.components_

In [ ]:
def top_words_for_topic(topic_index, n=12):
    weights = topic_word[topic_index]
    top_indices = weights.argsort()[::-1][:n]
    return pd.DataFrame({
        'topic': topic_index,
        'term': terms[top_indices],
        'weight': weights[top_indices]
    })

pd.concat([top_words_for_topic(k, n=10) for k in range(n_topics)], ignore_index=True)

## 2b. Topic-word weight plots

Top-word tables are easier to interpret when paired with the relative weights inside each topic.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7), sharex=False)
axes = axes.ravel()

for topic_index, ax in enumerate(axes[:n_topics]):
    topic_terms = top_words_for_topic(topic_index, n=8).sort_values('weight')
    ax.barh(topic_terms['term'], topic_terms['weight'], color='#4C72B0')
    ax.set_title(f'Topic {topic_index}')
    ax.set_xlabel('Weight')

for ax in axes[n_topics:]:
    ax.axis('off')

plt.tight_layout()

## 3. Representative documents

Top words are not enough. Representative documents help decide whether the topic label is defensible.

In [ ]:
topic_cols = [f'topic_{k}' for k in range(n_topics)]
doc_topic_df = pd.DataFrame(doc_topic, columns=topic_cols)
topic_docs = pd.concat([docs[['doc_id', 'category', 'broad_area', 'text']], doc_topic_df], axis=1)


def representative_docs(topic_index, n=5):
    col = f'topic_{topic_index}'
    return (
        topic_docs
        .sort_values(col, ascending=False)
        [['doc_id', 'category', 'broad_area', col, 'text']]
        .head(n)
    )

representative_docs(0, n=5)

Change the topic number in the next cell and inspect the documents. Do the documents support the label you would assign from top words alone?

In [ ]:
representative_docs(3, n=5)

## 4. Topic prevalence by metadata

Topic output becomes more useful when connected to document metadata. Because the model never saw the newsgroup category labels, category-topic alignment is a validation check rather than a supervised target.

In [ ]:
prevalence_by_area = topic_docs.groupby('broad_area')[topic_cols].mean().sort_index()
prevalence_by_category = topic_docs.groupby('category')[topic_cols].mean().sort_index()

prevalence_by_area

In [ ]:
prevalence_by_area.T.plot(kind='bar', figsize=(9, 4))
plt.title('Average topic prevalence by broad source area')
plt.ylabel('Mean topic proportion')
plt.tight_layout()

## 4a. Topic prevalence by source category

This view asks whether unsupervised topics align with known source categories. Strong alignment can aid interpretation, while weak or surprising alignment is a cue for closer document inspection.

In [ ]:
plt.figure(figsize=(9, 5))
sns.heatmap(prevalence_by_category, cmap='mako', cbar_kws={'label': 'Mean topic proportion'})
plt.title('Topic prevalence by newsgroup category')
plt.xlabel('Topic')
plt.ylabel('Category')
plt.tight_layout()

## 4b. STM-like topic prevalence regressions

A full Structural Topic Model estimates topics and covariate effects jointly. Here we do a lighter classroom version: estimate LDA first, then regress each document's estimated topic proportion on document metadata. This does not replace STM, but it makes the prevalence question concrete.

For each topic $k$:

$$
\hat\theta_{dk} = \alpha_k + \gamma_{k,science}\mathbb{1}(science_d) + \gamma_{k,politics}\mathbb{1}(politics_d) + \cdots + \epsilon_{dk}.
$$

The coefficients show how much more or less prevalent a topic is in each broad area relative to a baseline area.


In [ ]:
baseline_area = 'computing' if 'computing' in topic_docs['broad_area'].unique() else sorted(topic_docs['broad_area'].unique())[0]
area_dummies = pd.get_dummies(topic_docs['broad_area']).astype(float)
comparison_areas = [area for area in sorted(area_dummies.columns) if area != baseline_area]

design = pd.DataFrame({'intercept': 1.0}, index=topic_docs.index)
for area in comparison_areas:
    design[f'area: {area}'] = area_dummies[area]

prevalence_coef_rows = []
for topic in topic_cols:
    beta, *_ = np.linalg.lstsq(design.to_numpy(), topic_docs[topic].to_numpy(), rcond=None)
    for term, estimate in zip(design.columns, beta):
        prevalence_coef_rows.append({
            'topic': topic,
            'term': term,
            'estimate': estimate,
        })

prevalence_effects = pd.DataFrame(prevalence_coef_rows)
prevalence_effect_matrix = (
    prevalence_effects[prevalence_effects['term'] != 'intercept']
    .pivot(index='topic', columns='term', values='estimate')
    .sort_index()
)

plt.figure(figsize=(8, 5))
sns.heatmap(
    prevalence_effect_matrix,
    cmap='vlag',
    center=0,
    annot=True,
    fmt='.2f',
    cbar_kws={'label': f'Difference in prevalence vs. {baseline_area}'},
)
plt.title('STM-like prevalence regressions after LDA')
plt.xlabel('Document metadata covariate')
plt.ylabel('Topic')
plt.tight_layout()

prevalence_effect_matrix


Read this as a validation and interpretation tool, not as causal evidence. The metadata was not used to estimate the LDA model, so strong differences can help interpret topics. But because the regression is post-hoc, it does not model uncertainty in the topic estimates and it does not prove that broad area causes topic prevalence.


## 4c. Topic similarity

Topics are often not cleanly separated. A topic-similarity heatmap helps detect redundant topics.

In [ ]:
topic_similarity = cosine_similarity(topic_word)

plt.figure(figsize=(6, 5))
sns.heatmap(topic_similarity, annot=True, fmt='.2f', cmap='viridis', vmin=0, vmax=1)
plt.title('Cosine similarity between topic-word distributions')
plt.xlabel('Topic')
plt.ylabel('Topic')
plt.tight_layout()

### Additional demo: topic stability across random seeds

Topic models can look authoritative even when some topics are unstable. This cell refits the same model with several random seeds and asks how well each reference topic can be matched by top-word overlap.

In [ ]:
def topic_top_term_sets(components, n_terms=12):
    return [set(terms[row.argsort()[::-1][:n_terms]]) for row in components]

reference_sets = topic_top_term_sets(topic_word, n_terms=12)
stability_rows = []

for seed in [7, 21, 99]:
    seed_model = LatentDirichletAllocation(
        n_components=n_topics,
        learning_method='batch',
        max_iter=8,
        random_state=seed
    )
    seed_model.fit(X)
    seed_sets = topic_top_term_sets(seed_model.components_, n_terms=12)

    for ref_topic, ref_terms in enumerate(reference_sets):
        overlaps = []
        for candidate_topic, candidate_terms in enumerate(seed_sets):
            union = ref_terms | candidate_terms
            overlap = len(ref_terms & candidate_terms) / len(union)
            overlaps.append((candidate_topic, overlap))
        best_topic, best_jaccard = max(overlaps, key=lambda item: item[1])
        stability_rows.append({
            'seed': seed,
            'reference_topic': ref_topic,
            'best_matching_topic': best_topic,
            'best_jaccard': best_jaccard
        })

stability = pd.DataFrame(stability_rows)
stability_matrix = stability.pivot(index='seed', columns='reference_topic', values='best_jaccard')

plt.figure(figsize=(8, 3.5))
sns.heatmap(stability_matrix, annot=True, fmt='.2f', cmap='YlGnBu', vmin=0, vmax=1)
plt.title('Topic stability: best top-word overlap across seeds')
plt.xlabel('Reference topic from main model')
plt.ylabel('Alternative random seed')
plt.tight_layout()

stability.groupby('reference_topic')['best_jaccard'].mean().sort_values().to_frame('mean_best_jaccard')

## 5. Compare topic counts

There is no universally correct number of topics. Compare several models and inspect whether the extra topics are useful.

In [ ]:
comparison_rows = []

for k in [5, 8, 12]:
    model = LatentDirichletAllocation(n_components=k, max_iter=12, random_state=42, learning_method='batch')
    model.fit(X)
    comparison_rows.append({
        'topics': k,
        'perplexity_lower_is_better': model.perplexity(X)
    })

comparison = pd.DataFrame(comparison_rows)
comparison

## 5a. Model-comparison plot

Perplexity is not validity, but plotting it makes clear that fit diagnostics and interpretability need to be considered separately.

In [ ]:
plt.figure(figsize=(5, 4))
sns.lineplot(data=comparison, x='topics', y='perplexity_lower_is_better', marker='o')
plt.title('Topic-count comparison')
plt.ylabel('Perplexity (lower is better)')
plt.tight_layout()

Perplexity is a model-fit diagnostic, not a validity test. A lower value does not guarantee more interpretable or more useful topics.

## 6. A lightweight projection exercise

The slides discuss Wordfish as a text-scaling model. This notebook uses SVD as a lightweight proxy for the same broad idea: placing documents in a lower-dimensional space.

SVD factorizes the document-term matrix into document scores, singular values, and term loadings:

$$
X \approx U_r \Sigma_r V_r^\top.
$$

The document coordinates are $U_r\Sigma_r$. Unlike Wordfish, SVD is not a Poisson count model and does not estimate word discrimination parameters. It simply finds orthogonal directions that summarize variation in the TF-IDF matrix. The sign of a component is arbitrary: multiplying both document scores and term loadings by $-1$ gives the same solution.


In [ ]:
tfidf = TfidfVectorizer(analyzer=lambda tokens: tokens, min_df=4, max_df=0.70, max_features=3500)
Z = tfidf.fit_transform(docs['tokens'])

svd = TruncatedSVD(n_components=3, random_state=42)
svd_coordinates = svd.fit_transform(Z)

# Component 1 often captures broad commonness. Component 2 is usually more contrastive,
# so it is more useful for a one-dimensional scaling illustration.
docs['svd_commonness'] = svd_coordinates[:, 0]
docs['dimension_1'] = svd_coordinates[:, 1]

svd_explained = pd.DataFrame({
    'component': ['component 1', 'component 2', 'component 3'],
    'explained_variance_ratio': svd.explained_variance_ratio_,
})

display(svd_explained)
docs.groupby('category')['dimension_1'].agg(['count', 'mean', 'std']).sort_values('mean')


## 6a. Interpreting an SVD dimension

A one-dimensional projection is only useful if we can explain what separates high-scoring from low-scoring documents. Inspecting term loadings helps, but the interpretation still needs substantive validation against documents and metadata.


In [ ]:
term_loadings = pd.DataFrame({
    'term': tfidf.get_feature_names_out(),
    'loading': svd.components_[1],
})

extreme_terms = pd.concat([
    term_loadings.nsmallest(12, 'loading'),
    term_loadings.nlargest(12, 'loading'),
]).sort_values('loading')
extreme_terms['direction'] = np.where(extreme_terms['loading'] > 0, 'positive loading', 'negative loading')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.barplot(data=svd_explained, x='component', y='explained_variance_ratio', color='#4C72B0', ax=axes[0])
axes[0].set_title('Variance summarized by SVD components')
axes[0].set_xlabel('')
axes[0].set_ylabel('Explained variance ratio')
axes[0].tick_params(axis='x', rotation=20)

sns.barplot(data=extreme_terms, x='loading', y='term', hue='direction', dodge=False, ax=axes[1])
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Terms defining SVD component 2')
axes[1].set_xlabel('Component loading')
axes[1].set_ylabel('')
axes[1].legend(loc='lower right')

plt.tight_layout()


## 6b. Document map with dominant topics

A two-dimensional map helps detect whether topic structure tracks category, broader area, or something else.

In [ ]:
svd_map = TruncatedSVD(n_components=2, random_state=42)
map_coords = svd_map.fit_transform(Z)

docs['map_x'] = map_coords[:, 0]
docs['map_y'] = map_coords[:, 1]
docs['dominant_topic'] = doc_topic.argmax(axis=1).astype(str)

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=docs,
    x='map_x',
    y='map_y',
    hue='category',
    style='dominant_topic',
    alpha=0.75,
    s=45
)
plt.title('SVD map of 20 Newsgroups posts, shaped by dominant topic')
plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()

In [ ]:
plt.figure(figsize=(8, 4.5))
sns.boxplot(data=docs, x='dimension_1', y='category', color='#4C72B0')
plt.title('One-dimensional projection by source category')
plt.xlabel('SVD dimension 1')
plt.ylabel('')
plt.tight_layout()

## 7. Validation memo template

Fill this in after inspecting at least one topic or cluster.

- Model estimated:
- Topic/cluster inspected:
- Tentative label:
- Evidence supporting the label:
- Evidence against the label:
- Possible artifact:
- Next validation check:

In [ ]:
# Your turn: choose a topic, inspect top words and representative documents, then fill in the memo above.
chosen_topic = 0
display(top_words_for_topic(chosen_topic, n=15))
representative_docs(chosen_topic, n=5)